# Before Start
> I use data from this: [kaggle - titanic](https://www.kaggle.com/competitions/titanic/data)

> Download the test and train datasets and put them in the current folder.

> ## The objective
The objective is to predict whether a passenger survived or not (0 = Died, 1 = Survived) based on features like age, sex, ticket class, fare, etc. <br>
It's a classic binary classification problem in machine learning.

> ## First (Step 1) -- Explore The Data (EDA),
 I started by reading the file and looking for information about the dataset.
(I renamed train.csv into titanic.csv)


---

In [46]:
import pandas as pd
import numpy as np

df = pd.read_csv("titanic.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [47]:
print(df.shape)
print(df.dtypes)



(891, 12)
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


In [48]:
print(df.isnull().sum())


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [49]:
print(df.describe)

<bound method NDFrame.describe of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                 

> ## Step 2 -- Drop Useless Columns
due to some columns like unique number column or the column which have many missing value could not use for machine learning model so that we will drop it.

---


In [50]:
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


> ## Step 3 -- Handle missing value.
From step 1,2.
They have 2 features to deal with "Age" and "Embarked"
---




In [51]:
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)

embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)

df.isnull().sum() # Check that all have 0 missing value


,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


> ## Step 4 -- Encode Categorical Variables.
Convert "Sex" to numbers: male = 0, female = 1. <br>
And one-hot encode "Embarked" (S, C, Q: 3 new columns)
---




In [52]:
sex_mapping = {"male" : 0, "female" : 1}
df["Sex"] = df["Sex"].map(sex_mapping)

df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,False,True
1,1,1,1,38.0,1,0,71.2833,False,False
2,1,3,1,26.0,0,0,7.9250,False,True
3,1,1,1,35.0,1,0,53.1000,False,True
4,0,3,0,35.0,0,0,8.0500,False,True


> ## Step 5 -- Feature Engineering. (Optional)

---




In [53]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1 # +1 counts the passenger themselves.
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df = df.drop(columns=["SibSp", "Parch"])
df

,Survived,Pclass,Sex,Age,Fare,Embarked_Q,Embarked_S,FamilySize,IsAlone
0,0,3,0,22.0,7.2500,False,True,2,0
1,1,1,1,38.0,71.2833,False,False,2,0
2,1,3,1,26.0,7.9250,False,True,1,1
3,1,1,1,35.0,53.1000,False,True,2,0
4,0,3,0,35.0,8.0500,False,True,1,1
...,...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,13.0000,False,True,1,1
887,1,1,1,19.0,30.0000,False,True,1,1
888,0,3,1,28.0,23.4500,False,True,4,0
889,1,1,0,26.0,30.0000,False,False,1,1


> ## Step 6 -- Seperate Features and Target

---




In [54]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

print(X.shape)
print(y.shape)


(891, 8)
(891,)


> ## Step 7 -- Split into Train and Test

---




In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(712, 8)
(179, 8)


> ## Step 8 -- Scale Numeric Features

---




In [56]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train) # learn + apply on train
X_test = scaler.transform(X_test) # only apply on test

> ## Step 9 -- Verify and Train a Model
(Apply - Feed data into Machine Learning Model)

---




In [59]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8212290502793296
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.72      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



> ## Full Pipeline Version
The pipeline starts at Step 8 (Scale Numeric Features) — it combines Step 8 and Step 9 into one single workflow. <br><br>
So instead of doing Step 8 and Step 9 separately, the Full Pipeline Version chains them together and does both in one .fit() call.

---




In [61]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

pipe.fit(X_train, y_train)
print("Pipeline Accuracy:", accuracy_score(y_test, pipe.predict(X_test)))

Pipeline Accuracy: 0.8212290502793296
